# Tuto how to train a GNN

## First method: Inductive Training

1. Import stuff

In [11]:
import os
import json
import numpy as np
import torch
from torch_geometric.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch_geometric.nn as pyg_nn

from models.dataset_inductive import GraphDatasetInductive
from models.model_utils import train_inductive, test_inductive

2. import your model (that you defined in the "/models" file)

In [12]:
from models.base_gat_combined import GATClassifierCombined


3. Define your parameters

In [13]:
# Paramaeters
json_dir          = './json_output/'
# Model parameters
in_channels       = 1280
hidden_channels   = 64
out_channels      = 64
num_layers        = 2
dropout           = 0.1
act               = 'relu'
# Training Prameters
lr                = 1e-2
weight_decay      = 5e-4
batch_size        = 4
epochs            = 10

device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

4. Load data and split it into training/Test dataset

In [14]:
# Load full dataset
full_dataset = GraphDatasetInductive(json_dir)

# Compute split sizes
n_total = len(full_dataset)
n_train = int(0.8 * n_total)
n_val   = int(0.1 * n_total)
n_test  = n_total - n_train - n_val  # ensure it adds up

# Split dataset
train_ds, val_ds, test_ds = random_split(full_dataset, [n_train, n_val, n_test])

# Create data loaders
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)


5. Initialize your model

In [15]:
# model, optimizer, loss
model     = GATClassifierCombined(in_channels=in_channels, 
                            hidden_channels=hidden_channels, 
                            out_channels=out_channels, 
                            num_layers=num_layers, 
                            dropout=dropout, 
                            act=act).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.BCELoss()

6. Train your model

In [16]:
torch.cuda.empty_cache()
# training loop
for epoch in range(1, epochs+1):

    train_loss = train_inductive(model, train_loader, optimizer, criterion, device=device)

    val_n_acc,  val_e_acc  = test_inductive(model, val_loader,  device=device)

    print(f'Epoch {epoch:02d} | '
          f'Loss: {train_loss:.4f} | '
          f'Val Node Acc:  {val_n_acc:.4f} | Val Edge Acc:  {val_e_acc:.4f}')
    
    
torch.save(model.state_dict(), 'model.pth')


Loss for mini-batch 0: 1.5824470520019531
Loss for mini-batch 1: 2.3748388290405273
Loss for mini-batch 2: 1.0583224296569824
Loss for mini-batch 3: 1.2556291818618774
Loss for mini-batch 4: 0.9707875847816467
Loss for mini-batch 5: 1.3632211685180664
Loss for mini-batch 6: 1.0695792436599731
Loss for mini-batch 7: 0.9671151638031006
Loss for mini-batch 8: 1.2822028398513794
Loss for mini-batch 9: 1.4416502714157104
Loss for mini-batch 10: 1.0087167024612427
Loss for mini-batch 11: 0.9979082345962524
Loss for mini-batch 12: 0.9401784539222717
Loss for mini-batch 13: 1.1206783056259155
Loss for mini-batch 14: 1.844947338104248
Loss for mini-batch 15: 0.9903539419174194
Loss for mini-batch 16: 1.2104384899139404
Loss for mini-batch 17: 1.164919376373291
Loss for mini-batch 18: 1.066998839378357
Loss for mini-batch 19: 1.2203843593597412
Correct nodes for mini-batch 0: 779
Correct edges for mini-batch 0: 446064
Correct nodes for mini-batch 1: 1449
Correct edges for mini-batch 1: 993096
Co

7. Test model

In [17]:
test_n_acc,  test_e_acc  = test_inductive(model, test_loader,  device=device)
print(f'Test Node Acc:  {test_n_acc:.4f} | Test Edge Acc:  {test_e_acc:.4f}')

Correct nodes for mini-batch 0: 548
Correct edges for mini-batch 0: 297810
Correct nodes for mini-batch 1: 929
Correct edges for mini-batch 1: 449708
Correct nodes for mini-batch 2: 1240
Correct edges for mini-batch 2: 608702
Test Node Acc:  0.5277 | Test Edge Acc:  0.8974


## Second method: Transductive Training

1. Import stuff

In [18]:
import os
import json
import numpy as np
import torch
from torch_geometric.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch_geometric.nn as pyg_nn

from models.dataset_transductive import GraphDatasetTransductive # This changed
from models.model_utils import train_combined, test_combined # This changed

2. import your model (that you defined in the "/models" file)

In [19]:
from models.base_gat_combined import GATClassifierCombined


3. Define your parameters

In [20]:
# Paramaeters
json_dir          = './json_output/'
# Model parameters
in_channels       = 1280
hidden_channels   = 256
out_channels      = 256
num_layers        = 5
dropout           = 0.1
act               = 'relu'
# Training Prameters
lr                = 1e-2
weight_decay      = 5e-4
batch_size        = 2
epochs            = 10

device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

4. Load data and split it into training/Test dataset

In [21]:
# Initialise dataset and dataloader
dataset = GraphDatasetTransductive(json_dir, struct_thresh=0.6, textural_thresh=0.4)  # This changed

loader = DataLoader(dataset, batch_size=batch_size, shuffle=True) # This changed

5. Initialize your model

In [22]:
# Initialize model
model = GATClassifierCombined(in_channels=in_channels, 
                            hidden_channels=hidden_channels, 
                            out_channels=out_channels, 
                            num_layers=num_layers, 
                            dropout=dropout, 
                            act=act).to(device)

# Loss function and Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = torch.nn.BCELoss()

6. Train your model

In [23]:
# training loop    # This changed
for epoch in range(1, epochs+1):
    loss = train_combined(model, optimizer, criterion, loader, node_loss_weight=1.0, edge_loss_weight=1.0)
    train_acc_struct, test_acc_struct, train_acc_coplanarity, test_acc_coplanarity = test_combined(model, criterion, loader,
                                                                                                   threshold_structural=0.5,
                                                                                                   threshold_coplanarity=0.5)
    
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')
    print(f'Structural/Textural: Train Acc: {train_acc_struct:.4f}, Test Acc: {test_acc_struct:.4f}')
    print(f'Coplanarity: Train Acc: {train_acc_coplanarity:.4f}, Test Acc: {test_acc_coplanarity:.4f}')

ValueError: need at least one array to concatenate